In [62]:
import pandas as pd
df=pd.read_csv('./Dataset_03.csv')

In [63]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,venue
0,2,Australia,Sri Lanka,0.1,0,0,NaN,Melbourne Cricket Ground
1,2,Australia,Sri Lanka,0.2,0,0,NaN,Melbourne Cricket Ground
2,2,Australia,Sri Lanka,0.3,1,0,NaN,Melbourne Cricket Ground
3,2,Australia,Sri Lanka,0.4,2,0,NaN,Melbourne Cricket Ground
4,2,Australia,Sri Lanka,0.5,0,0,NaN,Melbourne Cricket Ground
...,...,...,...,...,...,...,...,...
63883,964,Sri Lanka,Australia,19.3,1,0,Colombo,R Premadasa Stadium
63884,964,Sri Lanka,Australia,19.4,0,0,Colombo,R Premadasa Stadium
63885,964,Sri Lanka,Australia,19.5,0,DM de Silva,Colombo,R Premadasa Stadium
63886,964,Sri Lanka,Australia,19.6,2,0,Colombo,R Premadasa Stadium


In [5]:
#performing EDA 
from pandas_profiling import ProfileReport
prof=ProfileReport(df)
prof.to_file(output_file='EDA.html')

C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\1185944488.py:2: DeprecationWarning: `import pandas_profiling` is going to be deprecated by April 1st. Please use `import ydata_profiling` instead.
  from pandas_profiling import ProfileReport


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:01<00:00,  7.57it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [64]:
df.isnull().sum()

match_id               0
batting_team           0
bowling_team           0
ball                   0
runs                   0
player_dismissed       0
city                8548
venue                  0
dtype: int64

In [65]:
#There are many null values in the city feature.

In [66]:
#Incase of all missing city we can use the venue's first letter as it will be the city. 


In [67]:
df[df['city'].isnull()]['venue'].value_counts()

venue
Dubai International Cricket Stadium        2969
Pallekele International Cricket Stadium    2066
Melbourne Cricket Ground                   1453
Sydney Cricket Ground                       749
Adelaide Oval                               498
Harare Sports Club                          372
Sharjah Cricket Stadium                     249
Sylhet International Cricket Stadium        128
Carrara Oval                                 64
Name: count, dtype: int64

In [68]:
#Using numpy for this feature transformation.

In [69]:
# It split the venue into words from which we need only first word to use it as city.

In [70]:
df['venue'].str.split()

0        [Melbourne, Cricket, Ground]
1        [Melbourne, Cricket, Ground]
2        [Melbourne, Cricket, Ground]
3        [Melbourne, Cricket, Ground]
4        [Melbourne, Cricket, Ground]
                     ...             
63883         [R, Premadasa, Stadium]
63884         [R, Premadasa, Stadium]
63885         [R, Premadasa, Stadium]
63886         [R, Premadasa, Stadium]
63887         [R, Premadasa, Stadium]
Name: venue, Length: 63888, dtype: object

In [71]:
df['venue'].str.split().apply(lambda x:x[0])

0        Melbourne
1        Melbourne
2        Melbourne
3        Melbourne
4        Melbourne
           ...    
63883            R
63884            R
63885            R
63886            R
63887            R
Name: venue, Length: 63888, dtype: object

In [72]:
import numpy as np
cities=np.where(df['city'].isnull(),df['venue'].str.split().apply(lambda x:x[0]),df['city'])

In [73]:
df['city']=cities

In [74]:
df.isnull().sum()

match_id            0
batting_team        0
bowling_team        0
ball                0
runs                0
player_dismissed    0
city                0
venue               0
dtype: int64

In [75]:
#we not need venue as we have city, so we can drop it.

In [76]:
df.drop(columns='venue',inplace=True)

In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63888 entries, 0 to 63887
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   match_id          63888 non-null  int64  
 1   batting_team      63888 non-null  object 
 2   bowling_team      63888 non-null  object 
 3   ball              63888 non-null  float64
 4   runs              63888 non-null  int64  
 5   player_dismissed  63888 non-null  object 
 6   city              63888 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 3.4+ MB


In [78]:
#Now there are many cities so we are filtering those cities where atleast 5 matches were played that means in dataframe it's count greater than equal to 5*20*6=600. 

In [79]:
eligible_cities=df['city'].value_counts()[df['city'].value_counts() >600].index.tolist()

In [80]:
df=df[df['city'].isin(eligible_cities)]

In [81]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne
...,...,...,...,...,...,...,...
63883,964,Sri Lanka,Australia,19.3,1,0,Colombo
63884,964,Sri Lanka,Australia,19.4,0,0,Colombo
63885,964,Sri Lanka,Australia,19.5,0,DM de Silva,Colombo
63886,964,Sri Lanka,Australia,19.6,2,0,Colombo


In [82]:
#For current run we need cumulative sum as after 12 balls what is score ,is runs cumulative sum.

In [83]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50501 entries, 0 to 63887
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   match_id          50501 non-null  int64  
 1   batting_team      50501 non-null  object 
 2   bowling_team      50501 non-null  object 
 3   ball              50501 non-null  float64
 4   runs              50501 non-null  int64  
 5   player_dismissed  50501 non-null  object 
 6   city              50501 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 3.1+ MB


In [84]:
df['current_score']=df.groupby('match_id')['runs'].cumsum()

C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\2789925231.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['current_score']=df.groupby('match_id')['runs'].cumsum()


In [85]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3
...,...,...,...,...,...,...,...,...
63883,964,Sri Lanka,Australia,19.3,1,0,Colombo,125
63884,964,Sri Lanka,Australia,19.4,0,0,Colombo,125
63885,964,Sri Lanka,Australia,19.5,0,DM de Silva,Colombo,125
63886,964,Sri Lanka,Australia,19.6,2,0,Colombo,127


In [86]:
# we need how many balls left.


In [87]:
#For this we need to separate over and ball from balls feature as 7.5 means 8th over ki 5 vi ball.

In [88]:
df['over']=df['ball'].apply(lambda x:str(x).split(".")[0])
df['ball_no']=df['ball'].apply(lambda x:str(x).split(".")[1])

C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\788310191.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['over']=df['ball'].apply(lambda x:str(x).split(".")[0])
C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\788310191.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['ball_no']=df['ball'].apply(lambda x:str(x).split(".")[1])


In [89]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score,over,ball_no
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,0,1
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,0,2
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1,0,3
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3,0,4
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3,0,5
...,...,...,...,...,...,...,...,...,...,...
63883,964,Sri Lanka,Australia,19.3,1,0,Colombo,125,19,3
63884,964,Sri Lanka,Australia,19.4,0,0,Colombo,125,19,4
63885,964,Sri Lanka,Australia,19.5,0,DM de Silva,Colombo,125,19,5
63886,964,Sri Lanka,Australia,19.6,2,0,Colombo,127,19,6


In [90]:
# Number of balls bowled after that ball.

In [91]:
df['balls_bowled']=(df['over'].astype('int')*6 + df['ball_no'].astype('int'))

C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\359639707.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['balls_bowled']=(df['over'].astype('int')*6 + df['ball_no'].astype('int'))


In [92]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score,over,ball_no,balls_bowled
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,0,1,1
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,0,2,2
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1,0,3,3
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3,0,4,4
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3,0,5,5
...,...,...,...,...,...,...,...,...,...,...,...
63883,964,Sri Lanka,Australia,19.3,1,0,Colombo,125,19,3,117
63884,964,Sri Lanka,Australia,19.4,0,0,Colombo,125,19,4,118
63885,964,Sri Lanka,Australia,19.5,0,DM de Silva,Colombo,125,19,5,119
63886,964,Sri Lanka,Australia,19.6,2,0,Colombo,127,19,6,120


In [93]:
#There will extras in over that lead to more than 120 balls in any particular inning.

In [94]:
#For balls left we substract from 120 but there will be negative also.

In [95]:
df['balls_left']=120-df['balls_bowled']
df['balls_left']=df['balls_left'].apply(lambda x:0 if x<0 else x)

C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\3502974883.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['balls_left']=120-df['balls_bowled']
C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\3502974883.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['balls_left']=df['balls_left'].apply(lambda x:0 if x<0 else x)


In [96]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score,over,ball_no,balls_bowled,balls_left
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,0,1,1,119
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,0,2,2,118
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1,0,3,3,117
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3,0,4,4,116
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3,0,5,5,115
...,...,...,...,...,...,...,...,...,...,...,...,...
63883,964,Sri Lanka,Australia,19.3,1,0,Colombo,125,19,3,117,3
63884,964,Sri Lanka,Australia,19.4,0,0,Colombo,125,19,4,118,2
63885,964,Sri Lanka,Australia,19.5,0,DM de Silva,Colombo,125,19,5,119,1
63886,964,Sri Lanka,Australia,19.6,2,0,Colombo,127,19,6,120,0


In [97]:
raw=df['player_dismissed']

In [98]:
raw

0                  0
1                  0
2                  0
3                  0
4                  0
            ...     
63883              0
63884              0
63885    DM de Silva
63886              0
63887              0
Name: player_dismissed, Length: 50501, dtype: object

In [99]:
# wickets left i can get it from player_dismissed feature as there is either 0 or name of player that out.

In [50]:
# I replace all that names with 1 and then apply cumulative sum by match id.

In [101]:
w=df['player_dismissed'].apply(lambda x:0 if x=='0' else 1)


0        0
1        0
2        0
3        0
4        0
        ..
63883    0
63884    0
63885    1
63886    0
63887    0
Name: player_dismissed, Length: 50501, dtype: int64

In [102]:
r=w

In [103]:
df['player_dismissed']=r

C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\1691413822.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['player_dismissed']=r


In [104]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score,over,ball_no,balls_bowled,balls_left
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,0,1,1,119
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,0,2,2,118
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1,0,3,3,117
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3,0,4,4,116
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3,0,5,5,115
...,...,...,...,...,...,...,...,...,...,...,...,...
63883,964,Sri Lanka,Australia,19.3,1,0,Colombo,125,19,3,117,3
63884,964,Sri Lanka,Australia,19.4,0,0,Colombo,125,19,4,118,2
63885,964,Sri Lanka,Australia,19.5,0,1,Colombo,125,19,5,119,1
63886,964,Sri Lanka,Australia,19.6,2,0,Colombo,127,19,6,120,0


In [105]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 50501 entries, 0 to 63887
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   match_id          50501 non-null  int64  
 1   batting_team      50501 non-null  object 
 2   bowling_team      50501 non-null  object 
 3   ball              50501 non-null  float64
 4   runs              50501 non-null  int64  
 5   player_dismissed  50501 non-null  int64  
 6   city              50501 non-null  object 
 7   current_score     50501 non-null  int64  
 8   over              50501 non-null  object 
 9   ball_no           50501 non-null  object 
 10  balls_bowled      50501 non-null  int64  
 11  balls_left        50501 non-null  int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 5.0+ MB


In [106]:
df['player_dismissed']=df.groupby('match_id')['player_dismissed'].cumsum()

C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\3889851133.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['player_dismissed']=df.groupby('match_id')['player_dismissed'].cumsum()


In [107]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score,over,ball_no,balls_bowled,balls_left
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,0,1,1,119
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,0,2,2,118
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1,0,3,3,117
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3,0,4,4,116
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3,0,5,5,115
...,...,...,...,...,...,...,...,...,...,...,...,...
63883,964,Sri Lanka,Australia,19.3,1,8,Colombo,125,19,3,117,3
63884,964,Sri Lanka,Australia,19.4,0,8,Colombo,125,19,4,118,2
63885,964,Sri Lanka,Australia,19.5,0,9,Colombo,125,19,5,119,1
63886,964,Sri Lanka,Australia,19.6,2,9,Colombo,127,19,6,120,0


In [108]:
df['wickets_left']=10-df['player_dismissed']

C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\867291440.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['wickets_left']=10-df['player_dismissed']


In [109]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score,over,ball_no,balls_bowled,balls_left,wickets_left
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,0,1,1,119,10
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,0,2,2,118,10
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1,0,3,3,117,10
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3,0,4,4,116,10
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3,0,5,5,115,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
63883,964,Sri Lanka,Australia,19.3,1,8,Colombo,125,19,3,117,3,2
63884,964,Sri Lanka,Australia,19.4,0,8,Colombo,125,19,4,118,2,2
63885,964,Sri Lanka,Australia,19.5,0,9,Colombo,125,19,5,119,1,1
63886,964,Sri Lanka,Australia,19.6,2,9,Colombo,127,19,6,120,0,1


In [110]:
df['crr']=(df['current_score']*6)/df['balls_bowled']

C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\1648461935.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['crr']=(df['current_score']*6)/df['balls_bowled']


In [111]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score,over,ball_no,balls_bowled,balls_left,wickets_left,crr
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,0,1,1,119,10,0.000000
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,0,2,2,118,10,0.000000
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1,0,3,3,117,10,2.000000
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3,0,4,4,116,10,4.500000
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3,0,5,5,115,10,3.600000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63883,964,Sri Lanka,Australia,19.3,1,8,Colombo,125,19,3,117,3,2,6.410256
63884,964,Sri Lanka,Australia,19.4,0,8,Colombo,125,19,4,118,2,2,6.355932
63885,964,Sri Lanka,Australia,19.5,0,9,Colombo,125,19,5,119,1,1,6.302521
63886,964,Sri Lanka,Australia,19.6,2,9,Colombo,127,19,6,120,0,1,6.350000


In [112]:
# Now for model i have to fetch the last 5 overs runs scored that i will perform from rolling function of python.

In [116]:
groups=df.groupby('match_id')

match_ids=df['match_id'].unique()
last_five=[]
for id in match_ids:
    last_five.extend(groups.get_group(id)['runs'].rolling(window=30).sum().values.tolist())

In [119]:
len(last_five)

50501

In [120]:
df['last_five']=last_five

C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\2265987530.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['last_five']=last_five


In [121]:
df

,match_id,batting_team,bowling_team,ball,runs,player_dismissed,city,current_score,over,ball_no,balls_bowled,balls_left,wickets_left,crr,last_five
0,2,Australia,Sri Lanka,0.1,0,0,Melbourne,0,0,1,1,119,10,0.000000,NaN
1,2,Australia,Sri Lanka,0.2,0,0,Melbourne,0,0,2,2,118,10,0.000000,NaN
2,2,Australia,Sri Lanka,0.3,1,0,Melbourne,1,0,3,3,117,10,2.000000,NaN
3,2,Australia,Sri Lanka,0.4,2,0,Melbourne,3,0,4,4,116,10,4.500000,NaN
4,2,Australia,Sri Lanka,0.5,0,0,Melbourne,3,0,5,5,115,10,3.600000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63883,964,Sri Lanka,Australia,19.3,1,8,Colombo,125,19,3,117,3,2,6.410256,32.0
63884,964,Sri Lanka,Australia,19.4,0,8,Colombo,125,19,4,118,2,2,6.355932,32.0
63885,964,Sri Lanka,Australia,19.5,0,9,Colombo,125,19,5,119,1,1,6.302521,32.0
63886,964,Sri Lanka,Australia,19.6,2,9,Colombo,127,19,6,120,0,1,6.350000,33.0


In [123]:
final_df=df.groupby('match_id')['runs'].sum().reset_index().merge(df,on='match_id')

In [126]:
final_df=final_df[['batting_team','bowling_team','city','current_score','balls_left','wickets_left','crr','last_five','runs_x']]

In [127]:
final_df

,batting_team,bowling_team,city,current_score,balls_left,wickets_left,crr,last_five,runs_x
0,Australia,Sri Lanka,Melbourne,0,119,10,0.000000,NaN,168
1,Australia,Sri Lanka,Melbourne,0,118,10,0.000000,NaN,168
2,Australia,Sri Lanka,Melbourne,1,117,10,2.000000,NaN,168
3,Australia,Sri Lanka,Melbourne,3,116,10,4.500000,NaN,168
4,Australia,Sri Lanka,Melbourne,3,115,10,3.600000,NaN,168
...,...,...,...,...,...,...,...,...,...
50496,Sri Lanka,Australia,Colombo,125,3,2,6.410256,32.0,128
50497,Sri Lanka,Australia,Colombo,125,2,2,6.355932,32.0,128
50498,Sri Lanka,Australia,Colombo,125,1,1,6.302521,32.0,128
50499,Sri Lanka,Australia,Colombo,127,0,1,6.350000,33.0,128


In [128]:
final_df.isnull().sum()

batting_team         0
bowling_team         0
city                 0
current_score        0
balls_left           0
wickets_left         0
crr                  0
last_five        12024
runs_x               0
dtype: int64

In [129]:
final_df.dropna(inplace=True)

C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_25928\2709626079.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.dropna(inplace=True)


In [130]:
final_df.isnull().sum()

batting_team     0
bowling_team     0
city             0
current_score    0
balls_left       0
wickets_left     0
crr              0
last_five        0
runs_x           0
dtype: int64

In [131]:
#All missing values are removed.

In [132]:
#For no bias in model we shuffle the records.

In [133]:
final_df=final_df.sample(final_df.shape[0])

In [134]:
final_df

,batting_team,bowling_team,city,current_score,balls_left,wickets_left,crr,last_five,runs_x
21357,England,West Indies,London,124,23,6,7.670103,29.0,161
47172,Afghanistan,Sri Lanka,Kolkata,47,60,7,4.700000,15.0,153
18314,Australia,India,Mumbai,77,62,8,7.965517,41.0,166
36680,New Zealand,Sri Lanka,Pallekele,59,44,3,4.657895,39.0,74
43802,New Zealand,South Africa,Durban,132,20,5,7.920000,33.0,151
...,...,...,...,...,...,...,...,...,...
14036,South Africa,England,Cape Town,154,12,7,8.555556,64.0,191
7628,Bangladesh,West Indies,Lauderhill,48,76,8,6.545455,35.0,171
21571,South Africa,India,Nottingham,72,55,8,6.646154,26.0,130
32512,India,Australia,Colombo,56,81,9,8.615385,46.0,140


In [135]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38477 entries, 21357 to 14017
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   batting_team   38477 non-null  object 
 1   bowling_team   38477 non-null  object 
 2   city           38477 non-null  object 
 3   current_score  38477 non-null  int64  
 4   balls_left     38477 non-null  int64  
 5   wickets_left   38477 non-null  int64  
 6   crr            38477 non-null  float64
 7   last_five      38477 non-null  float64
 8   runs_x         38477 non-null  int64  
dtypes: float64(2), int64(4), object(3)
memory usage: 2.9+ MB


In [136]:
final_df.to_csv('Dataset_04.csv',index=False)